# LabelEncoder 집중 연습 · 이전 대응표 보관하고 다시 꺼내기

**전용 복습 10문항 / 합성 CSV / 창작 문제입니다.** 전체 모의시험이 아니라 LabelEncoder에만 집중합니다. 모델링은 하지 않습니다.

CSV와 노트북을 같은 폴더에 두고 처음부터 실행하세요. 여기서는 인코딩의 학습/변환 차이에 집중하도록 학습용·검증용 CSV를 미리 나눴습니다. 실제 시험의 분할 방식에 대한 주장은 아닙니다.

- 세 컬럼: airline(항공사), fare_class(좌석 등급), route(노선)
- 결측치 없음. 검증 값은 모두 학습에 존재하지만 일부 범주는 검증에 등장하지 않습니다.
- LabelEncoder는 값을 정렬해 0부터 정수로 바꾸며 classes_에 대응표를 보관합니다. fit/fit_transform은 대응표를 학습하고, transform은 기존 표로 변환합니다.
- 기본 용도는 정답 y 인코딩입니다. 이번에는 사용자가 접한 모의문제의 여러 범주형 컬럼 인코딩 방식도 연습합니다. 피처의 숫자 코드는 실제 순서·크기를 뜻하지 않습니다.
- 4번에는 **일부러 잘못된 코드**가 주어집니다. 정상 학습 코드와 혼동하지 마세요.
- 힌트와 정답은 별도입니다. 숨은 shape 채점기는 없습니다.


## 01. 데이터 읽기

pandas를 pd로, sklearn.preprocessing의 LabelEncoder를 불러오세요. encoder_train.csv는 train, encoder_valid.csv는 valid에 읽으세요. 원본을 보존할 복사본 train_bad, valid_bad도 만드세요.

<details><summary>힌트 보기</summary>

read_csv(), copy()

</details>

In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
train = pd.read_csv("encoder_train.csv")
valid = pd.read_csv("encoder_valid.csv")
train_bad = train.copy()
valid_bad = valid.copy()

## 02. 범주 확인

세 컬럼의 학습/검증 고유값을 각각 정렬해서 출력하세요. 학습에는 있지만 검증에는 없는 airline 이름을 answer02에 문자열로 저장하세요.

용도: 검증에 일부 범주가 없더라도 학습 때 만든 번호를 유지해야 합니다.

<details><summary>힌트 보기</summary>

unique(), sorted()

</details>

In [2]:
for col in ["airline", "fare_class", "route"]:
    print(col, "train:", sorted(train[col].unique()), "valid:", sorted(valid[col].unique()))
answer02 = "Cloud"

airline train: ['Aero', 'Cloud', 'Sky'] valid: ['Aero', 'Sky']
fare_class train: ['Business', 'Economy', 'First'] valid: ['Business', 'First']
route train: ['East', 'North', 'West'] valid: ['North', 'West']


## 03. 첫 인코더 보관

LabelEncoder를 le에 생성해 train_bad의 airline을 fit_transform하세요. 번호를 airline에 다시 저장하세요. 이때 le를 saved_le에도 대입하고, 당시 대응표 classes_를 복사해 airline_classes_before에 저장하세요.

주의: saved_le = le는 객체 복사가 아닙니다. 같은 객체를 가리키는 이름을 하나 더 만드는 것입니다.

<details><summary>힌트 보기</summary>

saved_le = le, le.classes_.copy()

</details>

In [3]:
le = LabelEncoder()
train_bad["airline"] = le.fit_transform(train_bad["airline"])
saved_le = le
airline_classes_before = le.classes_.copy()
print(airline_classes_before)

['Aero' 'Cloud' 'Sky']


## 04. 같은 객체에 다시 fit하면?

아래 제공 코드를 실행하고, answer04에 **saved_le가 현재 보관한 것이 "airline"인지 "fare_class"인지** 저장하세요. saved_le.classes_와 airline_classes_before를 비교해 보세요.

이 코드는 실패 원인을 관찰하기 위한 예시입니다. `saved_le = le`를 해두어도 같은 객체에 다시 fit하면 이전 대응표는 보존되지 않습니다.

<details><summary>힌트 보기</summary>

saved_le is le는 True입니다. 기존 숫자 데이터는 그대로지만 객체의 대응표만 바뀝니다.

</details>

In [4]:
# 의도적으로 같은 객체를 재학습하는 잘못된 예시
train_bad["fare_class"] = le.fit_transform(train_bad["fare_class"])
print("같은 객체:", saved_le is le)
print("당시 항공사:", airline_classes_before)
print("현재 saved_le:", saved_le.classes_)
print("잘못 복원한 airline:", saved_le.inverse_transform(train_bad["airline"].head(5)))
answer04 = "fare_class"

같은 객체: True
당시 항공사: ['Aero' 'Cloud' 'Sky']
현재 saved_le: ['Business' 'Economy' 'First']
잘못 복원한 airline: ['Business' 'First' 'First' 'Economy' 'Economy']


## 05. 컬럼별 인코더를 독립 보관

원본 train을 복사해 train_encoded에 저장하세요. encoders = {}를 만들고 airline, fare_class, route를 반복 처리하세요. **반복문 안에서 매번 새 LabelEncoder를 생성**해 학습하고, encoders[col]에 인코더를 보관하세요. train_encoded의 해당 컬럼에는 변환한 번호를 저장하세요.

용도: 딕셔너리에 컬럼별 객체를 보관하면 나중에 해당 대응표를 다시 꺼낼 수 있습니다. 같은 객체를 계속 fit해 딕셔너리에 넣으면 모두 마지막 대응표를 공유합니다.

<details><summary>힌트 보기</summary>

for col in ...:
    encoder = LabelEncoder()
    ...
    encoders[col] = encoder

</details>

In [5]:
train_encoded = train.copy()
encoders = {}
for col in ["airline", "fare_class", "route"]:
    encoder = LabelEncoder()
    train_encoded[col] = encoder.fit_transform(train[col])
    encoders[col] = encoder
for col in encoders:
    print(col, encoders[col].classes_)

airline ['Aero' 'Cloud' 'Sky']
fare_class ['Business' 'Economy' 'First']
route ['East' 'North' 'West']


## 06. 이전 인코더 꺼내기

encoders에서 airline 인코더를 꺼내 le_airline에 저장하세요. 새로 fit하지 말고 valid의 airline을 transform하여 airline_valid_codes에 저장해 출력하세요.

문제에서 말하는 “이전 것을 불러오기”의 메모리 안 버전입니다. 학습한 객체를 다시 꺼내 쓰는 것입니다.

<details><summary>힌트 보기</summary>

le_airline = encoders["airline"], transform()

</details>

In [6]:
le_airline = encoders["airline"]
airline_valid_codes = le_airline.transform(valid["airline"])
print(airline_valid_codes[:10])

[2 2 2 0 0 0 0 0 0 2]


## 07. 검증 전체에 같은 대응표 적용

valid를 복사해 valid_encoded에 저장하세요. 세 컬럼 각각 encoders[col].transform으로 변환하세요. 검증에서는 fit이나 fit_transform을 사용하지 마세요. airline의 학습/검증 고유 번호를 출력하세요.

<details><summary>힌트 보기</summary>

for col ... transform(valid[col])

</details>

In [7]:
valid_encoded = valid.copy()
for col in ["airline", "fare_class", "route"]:
    valid_encoded[col] = encoders[col].transform(valid[col])
print("train airline:", sorted(train_encoded["airline"].unique()))
print("valid airline:", sorted(valid_encoded["airline"].unique()))

train airline: [np.int64(0), np.int64(1), np.int64(2)]
valid airline: [np.int64(0), np.int64(2)]


## 08. 번호를 원래 문자열로 복원

encoders의 airline 인코더로 valid_encoded의 airline을 inverse_transform하여 restored_airline에 저장하세요. valid의 원본 airline과 모든 값이 같은지 확인하세요.

용도: transform은 문자열→번호, inverse_transform은 번호→문자열입니다. 반드시 그 번호를 만든 대응표로 복원해야 합니다.

<details><summary>힌트 보기</summary>

inverse_transform(), (복원값 == 원본).all()

</details>

In [8]:
restored_airline = encoders["airline"].inverse_transform(valid_encoded["airline"])
print(restored_airline[:10])
print((restored_airline == valid["airline"]).all())

['Sky' 'Sky' 'Sky' 'Aero' 'Aero' 'Aero' 'Aero' 'Aero' 'Aero' 'Sky']
True


## 09. 파일에 저장하고 다시 불러오기

joblib을 불러오세요. encoders를 label_encoders.joblib에 dump하고, load한 결과를 loaded_encoders에 저장하세요. loaded_encoders의 route 인코더로 valid의 route를 transform하여 route_reloaded_codes에 저장하세요. valid_encoded의 route와 같은지 출력하세요.

용도: 메모리의 딕셔너리는 현재 실행 중에 쓰고, 파일 저장은 다른 세션에서도 같은 대응표를 쓰기 위한 것입니다.

<details><summary>힌트 보기</summary>

joblib.dump(객체, 경로), joblib.load(경로)

</details>

In [9]:
import joblib
joblib.dump(encoders, "label_encoders.joblib")
loaded_encoders = joblib.load("label_encoders.joblib")
route_reloaded_codes = loaded_encoders["route"].transform(valid["route"])
print((route_reloaded_codes == valid_encoded["route"]).all())

True


## 10. 검증 데이터에 fit하면 왜 틀릴까?

아래 관찰 코드를 실행하세요. 새 LabelEncoder를 valid의 airline에만 fit_transform한 bad_codes와 7번 정상 번호를 비교합니다. answer10에 **학습 기준에서 Sky의 번호**를, answer10_bad에 **검증에서 따로 fit한 Sky의 번호**를 저장하세요. 이 데이터에서 번호가 같아지는지 다른지 확인하세요.

중요: 검증에 새로운 문자열이 생겨 transform 오류가 나더라도 검증에 새로 fit하는 것은 해결책이 아닙니다. 이 세트에는 미등장 범주가 없으므로 학습 인코더의 transform만으로 처리됩니다.

<details><summary>힌트 보기</summary>

원래 train은 Aero, Cloud, Sky / valid는 Aero, Sky입니다.

</details>

In [10]:
bad_encoder = LabelEncoder()
bad_codes = bad_encoder.fit_transform(valid["airline"])
print("정상 번호:", valid_encoded["airline"].head(10).tolist())
print("잘못된 번호:", bad_codes[:10])
answer10 = int(encoders["airline"].transform(["Sky"])[0])
answer10_bad = int(bad_encoder.transform(["Sky"])[0])
print("학습 기준 Sky:", answer10)
print("검증 재학습 Sky:", answer10_bad)

정상 번호: [2, 2, 2, 0, 0, 0, 0, 0, 0, 2]
잘못된 번호: [1 1 1 0 0 0 0 0 0 1]
학습 기준 Sky: 2
검증 재학습 Sky: 1
